# ⏱️ Survival Analysis
**ISLP Chapter 11 · Pattern Recognition for the Rest of Us**

> Survival analysis answers questions like: how long until a patient relapses? When will a customer churn? It handles a unique challenge — censored observations: people who haven't experienced the event yet when the study ends.

### What you'll learn
- Censoring: why standard regression fails for time-to-event data
- Kaplan-Meier estimator: non-parametric survival curve
- Log-rank test: comparing survival between groups
- Cox proportional hazards model: regression for survival data
- Hazard ratio interpretation

### Dataset: ROSSI recidivism + simulated patient data
### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
!pip install lifelines -q
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.datasets import load_rossi
import warnings; warnings.filterwarnings('ignore')

# Rossi recidivism dataset: time until re-arrest after release from prison
rossi = load_rossi()
print(f'Rossi dataset: {rossi.shape}')
print(rossi.head())
print(f"\nEvent rate: {rossi['arrest'].mean():.1%} arrested")
print(f"Duration range: {rossi['week'].min()}–{rossi['week'].max()} weeks")

## 🎯 Part 1 — What is Censoring?

In time-to-event data:
- Some subjects experience the event (arrested, died, churned)
- Others are **censored** — we know they survived UNTIL a certain time, but not what happened after
  - Study ended before they experienced the event
  - Lost to follow-up
  - Withdrew from study

**Why standard regression fails:** A censored observation isn't a missing value — it contains real information (the person survived for at least X weeks). Ignoring it or treating it as missing biases estimates.

**Survival analysis** handles censoring properly.

In [ ]:
# Visualize censoring
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Show 20 individual timelines
np.random.seed(42)
sample_idx = np.random.choice(len(rossi), 25, replace=False)
for i, idx in enumerate(sample_idx):
    row = rossi.iloc[idx]
    color = '#e85d20' if row['arrest']==1 else '#1e5fa8'
    marker = 'x' if row['arrest']==1 else 'o'
    axes[0].plot([0, row['week']], [i, i], color=color, lw=1.5, alpha=0.8)
    axes[0].plot(row['week'], i, marker=marker, color=color, markersize=7)
axes[0].set_xlabel('Weeks'); axes[0].set_ylabel('Subject')
axes[0].set_title('Event timelines\n× = arrested (event), o = censored')
from matplotlib.lines import Line2D
legend = [Line2D([0],[0],color='#e85d20',marker='x',label='Arrested'),
          Line2D([0],[0],color='#1e5fa8',marker='o',label='Censored')]
axes[0].legend(handles=legend)

# Histogram of observation times by event status
for arrested, color, label in [(1,'#e85d20','Arrested'),(0,'#1e5fa8','Censored')]:
    subset = rossi[rossi['arrest']==arrested]['week']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=f'{label} (n={len(subset)})', edgecolor='white')
axes[1].set_xlabel('Weeks'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Follow-up Times')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Kaplan-Meier survival curve — overall
kmf = KaplanMeierFitter()
kmf.fit(rossi['week'], event_observed=rossi['arrest'], label='All subjects')

fig, ax = plt.subplots(figsize=(9, 5))
kmf.plot_survival_function(ax=ax, color='#1e5fa8', linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release')
ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier Survival Curve — Recidivism Study')
ax.axhline(0.5, color='#888', lw=1, ls='--', label='Median survival')
median = kmf.median_survival_time_
if not np.isinf(median):
    ax.axvline(median, color='#888', lw=1, ls='--')
    ax.text(median+1, 0.52, f'Median = {median:.0f} weeks', fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()
print(f'\nMedian survival time: {kmf.median_survival_time_:.1f} weeks')
print(f'52-week survival estimate: {kmf.survival_function_at_times([52]).values[0]:.3f}')

In [ ]:
# Compare groups: financial aid vs no financial aid
fig, ax = plt.subplots(figsize=(9, 5))
for group, color, label in [(1,'#1a7a45','Received financial aid'),(0,'#e85d20','No financial aid')]:
    kmf_g = KaplanMeierFitter()
    mask = rossi['fin']==group
    kmf_g.fit(rossi.loc[mask,'week'], event_observed=rossi.loc[mask,'arrest'], label=label)
    kmf_g.plot_survival_function(ax=ax, color=color, linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release'); ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier by Financial Aid Status')
ax.legend()
plt.tight_layout(); plt.show()

# Log-rank test
results = logrank_test(
    rossi.loc[rossi['fin']==1,'week'], rossi.loc[rossi['fin']==0,'week'],
    event_observed_A=rossi.loc[rossi['fin']==1,'arrest'],
    event_observed_B=rossi.loc[rossi['fin']==0,'arrest']
)
print(f'Log-rank test: p-value = {results.p_value:.4f}')
print('Financial aid significantly reduces recidivism' if results.p_value < 0.05 else 'No significant difference')

In [ ]:
# Cox Proportional Hazards Model
cox_features = ['fin','age','race','wexp','mar','paro','prio']
cph = CoxPHFitter()
cph.fit(rossi[cox_features + ['week','arrest']], duration_col='week', event_col='arrest')
cph.print_summary()

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title('Cox Model: Hazard Ratios (HR > 1 = higher risk of arrest)')
ax.axvline(0, color='black', lw=0.8, ls='--')
plt.tight_layout(); plt.show()
print('\n📌 HR < 0 (log scale) = protective factor, HR > 0 = risk factor')
print(f"   'prio' (prior arrests): HR = {np.exp(cph.params_['prio']):.2f} — each prior arrest increases risk by {(np.exp(cph.params_['prio'])-1)*100:.0f}%")

In [ ]:
# Exercise workspace
# Task 1: Compare survival curves by age group (< 30 vs >= 30)
# YOUR CODE HERE

# Task 2: Build a Cox model using only the 3 most significant predictors
# Compare its concordance index to the full model
# YOUR CODE HERE

# Task 3: Predict individual survival curves for two specific profiles:
# Profile A: young (25), prior arrests=0, financial aid
# Profile B: older (40), prior arrests=3, no financial aid
profile_A = pd.DataFrame({'fin':1,'age':25,'race':1,'wexp':1,'mar':0,'paro':1,'prio':0}, index=[0])
profile_B = pd.DataFrame({'fin':0,'age':40,'race':1,'wexp':0,'mar':0,'paro':0,'prio':3}, index=[0])
# YOUR CODE HERE

In [ ]:
answers = {
    'q1': '',  # What is censoring in survival analysis?
    'q2': '',  # What does the Kaplan-Meier curve show?
    'q3': '',  # What does the log-rank test compare?
    'q4': '',  # What does a hazard ratio of 2.0 mean in a Cox model?
    'q5': '',  # Why can't you use standard linear regression for time-to-event outcomes?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Survival Analysis"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


## 📚 Further Reading
- [ISLP Ch 11](https://intro-stat-learning.github.io/ISLP/) — Survival analysis
- [lifelines docs](https://lifelines.readthedocs.io/)
- [Cox model interpretation guide](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html)

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*